# MSCS 634 - Lab 4: Regression Techniques & Regularization
**Student Name:** Bijay Raj KC  
**Course Title:**  Advanced Big Data and Data Mining (MSCS-634-B01)  
**Assignment:** Lab 4 - Linear, Multiple, Polynomial, Ridge, and Lasso Regression Analysis  
**Date:** July 2026

In [ ]:
# Import essential libraries for handling data, fitting models, and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics to evaluate model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Dataset loading and data splitting
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Set visual style for clean, readable plots
sns.set_theme(style="whitegrid")
%matplotlib inline

: 

In [ ]:
# Loading the real Diabetes dataset directly from scikit-learn
diabetes = load_diabetes(as_frame=True)

# Separate independent features (X) and target variable (y)
X = diabetes.data
y = diabetes.target

# Display preview of feature data
print("--- Features Overview ---")
print(X.head())

print("\n--- Target Data Preview (Disease progression score) ---")
print(y.head())

# Check shape and dataset information (442 instances, 10 features)
print(f"\nDataset Dimensions: {X.shape[0]} rows and {X.shape[1]} columns")

# Verify missing values to handle
missing_vals = X.isnull().sum().sum()
print(f"Total missing values found in dataset: {missing_vals}")

# Quick summary statistics
X.describe()

In [ ]:
# Select 'bmi' as the single feature for Simple Linear Regression
single_feature = X[['bmi']]

# Split dataset into training set (80%) and test set (20%)
X_train_slr, X_test_slr, y_train, y_test = train_test_split(
    single_feature, y, test_size=0.2, random_state=42
)

# Instantiate and fit Simple Linear Regression
slr_model = LinearRegression()
slr_model.fit(X_train_slr, y_train)

# Generate predictions on the test set
y_pred_slr = slr_model.predict(X_test_slr)

# Custom evaluation function to calculate required metrics
def evaluate_model(y_true, y_pred, model_name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"=== {model_name} Performance ===")
    print(f"MAE:  {mae:.4f}")
    print(f"MSE:  {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}\n")
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

# Store metrics for final summary
results = {}
results["Simple Linear Regression"] = evaluate_model(y_test, y_pred_slr, "Simple Linear Regression")

# Plot the simple linear regression line against actual test data
plt.figure(figsize=(8, 5))
plt.scatter(X_test_slr, y_test, color='royalblue', alpha=0.7, label='Actual Data')
plt.plot(X_test_slr, y_pred_slr, color='crimson', linewidth=2, label='Regression Line')
plt.title('Simple Linear Regression (BMI vs Diabetes Progression)')
plt.xlabel('BMI (Standardized)')
plt.ylabel('Disease Progression')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Using all independent variables from dataset
X_train_mlr, X_test_mlr, y_train_mlr, y_test_mlr = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create and train the Multiple Linear Regression model
mlr_model = LinearRegression()
mlr_model.fit(X_train_mlr, y_train_mlr)

# Predict target values on test set
y_pred_mlr = mlr_model.predict(X_test_mlr)

# Compute performance metrics
results["Multiple Regression"] = evaluate_model(y_test_mlr, y_pred_mlr, "Multiple Regression")

# Plot Actual vs Predicted values
plt.figure(figsize=(7, 6))
plt.scatter(y_test_mlr, y_pred_mlr, color='darkgreen', alpha=0.6)
plt.plot([y_test_mlr.min(), y_test_mlr.max()], [y_test_mlr.min(), y_test_mlr.max()], 'r--', lw=2, label='Perfect Fit Line')
plt.title('Multiple Regression: Actual vs Predicted Values')
plt.xlabel('Actual Target Values')
plt.ylabel('Predicted Target Values')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Applying degree 2 polynomial features
poly_degree2 = PolynomialFeatures(degree=2, include_bias=False)

# Transform features into polynomial feature matrix
X_train_poly2 = poly_degree2.fit_transform(X_train_mlr)
X_test_poly2 = poly_degree2.transform(X_test_mlr)

# Train linear model on degree 2 polynomial data
poly_model2 = LinearRegression()
poly_model2.fit(X_train_poly2, y_train_mlr)

# Predict and evaluate
y_pred_poly2 = poly_model2.predict(X_test_poly2)
results["Polynomial Regression (Degree 2)"] = evaluate_model(y_test_mlr, y_pred_poly2, "Polynomial Regression (Degree 2)")

# Compare training vs testing performance across polynomial degrees to illustrate overfitting
degrees = [1, 2, 3]
train_rmse_list = []
test_rmse_list = []

for d in degrees:
    pf = PolynomialFeatures(degree=d, include_bias=False)
    X_tr_p = pf.fit_transform(X_train_mlr)
    X_te_p = pf.transform(X_test_mlr)
    
    m = LinearRegression()
    m.fit(X_tr_p, y_train_mlr)
    
    tr_pred = m.predict(X_tr_p)
    te_pred = m.predict(X_te_p)
    
    train_rmse_list.append(np.sqrt(mean_squared_error(y_train_mlr, tr_pred)))
    test_rmse_list.append(np.sqrt(mean_squared_error(y_test_mlr, te_pred)))

# Plotting overfitting behavior
plt.figure(figsize=(8, 5))
plt.plot(degrees, train_rmse_list, marker='o', label='Train RMSE', color='blue')
plt.plot(degrees, test_rmse_list, marker='o', label='Test RMSE', color='orange')
plt.title('Polynomial Degree vs. RMSE (Demonstrating Overfitting)')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE')
plt.xticks(degrees)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Standardize features before regularized regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_mlr)
X_test_scaled = scaler.transform(X_test_mlr)

# Set regularization parameter alpha
alpha_val = 1.0

# 1. Ridge Regression (L2 penalty)
ridge = Ridge(alpha=alpha_val)
ridge.fit(X_train_scaled, y_train_mlr)
y_pred_ridge = ridge.predict(X_test_scaled)
results["Ridge Regression (alpha=1.0)"] = evaluate_model(y_test_mlr, y_pred_ridge, "Ridge Regression")

# 2. Lasso Regression (L1 penalty)
lasso = Lasso(alpha=alpha_val)
lasso.fit(X_train_scaled, y_train_mlr)
y_pred_lasso = lasso.predict(X_test_scaled)
results["Lasso Regression (alpha=1.0)"] = evaluate_model(y_test_mlr, y_pred_lasso, "Lasso Regression")

# Visualizing side-by-side performance of Ridge and Lasso
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge plot
axes[0].scatter(y_test_mlr, y_pred_ridge, color='purple', alpha=0.6)
axes[0].plot([y_test_mlr.min(), y_test_mlr.max()], [y_test_mlr.min(), y_test_mlr.max()], 'r--', lw=2)
axes[0].set_title('Ridge Regression: Actual vs Predicted')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')

# Lasso plot
axes[1].scatter(y_test_mlr, y_pred_lasso, color='teal', alpha=0.6)
axes[1].plot([y_test_mlr.min(), y_test_mlr.max()], [y_test_mlr.min(), y_test_mlr.max()], 'r--', lw=2)
axes[1].set_title('Lasso Regression: Actual vs Predicted')
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')

plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
summary_df = pd.DataFrame(results).T
print("=== Complete Model Performance Comparison ===")
summary_df